# Census ACS Raw Data Quality

**Purpose.** Audit the files ingested for this provider and the corresponding `raw.*`
DuckDB tables before any normalization, blending, or analytical transformation.

This notebook covers the supplied files/tables, observation grain, date and geography
coverage, column types and meanings, missingness and suppression, duplicate/invalid
keys, numeric ranges, suspicious values, source limitations, and downstream readiness.

## Setup and provider rules

In [1]:
from pathlib import Path
import re
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "quoll.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "quoll.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

PROVIDER = 'census_acs'
TABLE_PATTERNS = ['census_acs5_%']
PRIMARY_PATTERNS = ['census_acs5_county_affordability_%', 'census_acs5_county_dp02_%', 'census_acs5_county_dp03_%', 'census_acs5_county_dp04_%', 'census_acs5_county_dp05_%', 'census_acs5_county_migration_%', 'census_acs5_county_population_%']
KEY_CANDIDATES = [['year', 'state', 'county']]
DATE_CANDIDATES = ['year']
GEO_CANDIDATES = ['state', 'county']
NUMERIC_HINTS = ['year', 'median_household_income', 'total_population', 'median_home_value']
SUPPRESSION_CODES = ['-222222222', '-333333333', '-555555555', '-666666666', '-888888888', '-999999999', '(X)', 'N', '-']

def matches(name, patterns):
    return any(re.fullmatch(pattern.replace("%", ".*"), name, flags=re.I) for pattern in patterns)

def qi(value):
    return '"' + value.replace('"', '""') + '"'

raw_tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'raw' ORDER BY table_name"
).df()["table_name"].tolist()
provider_tables = [name for name in raw_tables if matches(name, TABLE_PATTERNS)]
primary_tables = [name for name in provider_tables if matches(name, PRIMARY_PATTERNS)]
provider_tables, primary_tables

(['census_acs5_b25103_variable_dictionary_2015_2024',
  'census_acs5_b25132_failures_2015_2025',
  'census_acs5_b25132_variable_dictionary_2021_2024',
  'census_acs5_b25133_failures_2015_2025',
  'census_acs5_b25133_variable_dictionary_2021_2024',
  'census_acs5_b25134_failures_2015_2025',
  'census_acs5_b25134_variable_dictionary_2021_2024',
  'census_acs5_b25135_failures_2015_2025',
  'census_acs5_b25135_variable_dictionary_2021_2024',
  'census_acs5_b25141_failures_2015_2025',
  'census_acs5_b25141_variable_dictionary_2023_2024',
  'census_acs5_county_affordability_2012_2024',
  'census_acs5_county_b25103_2015_2024',
  'census_acs5_county_b25103_metadata_2015_2024',
  'census_acs5_county_b25132_2021_2024',
  'census_acs5_county_b25132_metadata_2021_2024',
  'census_acs5_county_b25133_2021_2024',
  'census_acs5_county_b25133_metadata_2021_2024',
  'census_acs5_county_b25134_2021_2024',
  'census_acs5_county_b25134_metadata_2021_2024',
  'census_acs5_county_b25135_2021_2024',
  'censu

## Files and tables supplied

In [2]:
file_inventory = con.execute(
    '''
    SELECT table_name, filename, source_folder, source_path,
           loaded_at, row_count, detected_columns,
           upstream_source_url, content_sha256
    FROM meta.files
    WHERE table_schema = 'raw'
    ORDER BY table_name
    '''
).df()
file_inventory = file_inventory.loc[file_inventory["table_name"].isin(provider_tables)]

table_rows = []
for table in provider_tables:
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    column_count = con.execute(
        "SELECT count(*) FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).fetchone()[0]
    table_rows.append({"table_name": table, "rows": row_count, "columns": column_count,
                       "primary_data_table": table in primary_tables})
table_inventory = pd.DataFrame(table_rows)
display(file_inventory)
display(table_inventory)

,table_name,filename,source_folder,source_path,loaded_at,row_count,detected_columns,upstream_source_url,content_sha256
0,census_acs5_b25103_variable_dictionary_2015_2024,census_acs5_b25103_variable_dictionary_2015_20...,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:34.523081+00:00,60,"[""year"", ""variable"", ""label"", ""concept"", ""pred...",None,b4c9e822408fe85a0158a674abf482f071b4197fea0678...
1,census_acs5_b25132_variable_dictionary_2021_2024,census_acs5_b25132_variable_dictionary_2021_20...,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:34.549434+00:00,72,"[""year"", ""variable"", ""label"", ""concept"", ""pred...",None,492442d996e798956af2f0e490e8a6ee20a8970c72d482...
2,census_acs5_b25133_variable_dictionary_2021_2024,census_acs5_b25133_variable_dictionary_2021_20...,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:34.572082+00:00,72,"[""year"", ""variable"", ""label"", ""concept"", ""pred...",None,d5bd4702f9a3aed1d356a7fde90ce38c645eae211a15a3...
3,census_acs5_b25134_variable_dictionary_2021_2024,census_acs5_b25134_variable_dictionary_2021_20...,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:34.597516+00:00,72,"[""year"", ""variable"", ""label"", ""concept"", ""pred...",None,8a776af7ae48008c6c441c039760c269b8f0c4d66b2a14...
4,census_acs5_b25135_variable_dictionary_2021_2024,census_acs5_b25135_variable_dictionary_2021_20...,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:34.623255+00:00,48,"[""year"", ""variable"", ""label"", ""concept"", ""pred...",None,3a427b3f21411e1c8c8ffe62f1a27baf0d4a4e1094d5a9...
5,census_acs5_b25141_variable_dictionary_2023_2024,census_acs5_b25141_variable_dictionary_2023_20...,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:34.647532+00:00,108,"[""year"", ""variable"", ""label"", ""concept"", ""pred...",None,2a1e170ca30b7fb68f58b32ba8bc04fda4fc8f2422c335...
6,census_acs5_county_affordability_2012_2024,census_acs5_county_affordability_2012_2024.csv,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:35.318667+00:00,41870,"[""year"", ""county_fips"", ""state"", ""county"", ""NA...",None,061087703da926b2f5ba296b3fa7a668f15568ac394ea5...
7,census_acs5_county_b25103_2015_2024,census_acs5_county_b25103_2015_2024.csv,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:35.488231+00:00,32208,"[""year"", ""county_fips"", ""state"", ""county"", ""NA...",None,b8758171151f48b68b78462d62015fa47fd052c96b9876...
8,census_acs5_county_b25103_metadata_2015_2024,census_acs5_county_b25103_metadata_2015_2024.csv,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:35.516682+00:00,10,"[""field"", ""value""]",None,b19e966662fc784381176fa82036431daf1083192efa35...
9,census_acs5_county_b25132_2021_2024,census_acs5_county_b25132_2021_2024.csv,acs,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:35.691430+00:00,12887,"[""year"", ""county_fips"", ""state"", ""county"", ""NA...",None,16ad68ca971bbf9a9a08073c13020e222b0f27b7484e87...


,table_name,rows,columns,primary_data_table
0,census_acs5_b25103_variable_dictionary_2015_2024,60,6,False
1,census_acs5_b25132_failures_2015_2025,7,4,False
2,census_acs5_b25132_variable_dictionary_2021_2024,72,6,False
3,census_acs5_b25133_failures_2015_2025,7,4,False
4,census_acs5_b25133_variable_dictionary_2021_2024,72,6,False
5,census_acs5_b25134_failures_2015_2025,7,4,False
6,census_acs5_b25134_variable_dictionary_2021_2024,72,6,False
7,census_acs5_b25135_failures_2015_2025,7,4,False
8,census_acs5_b25135_variable_dictionary_2021_2024,48,6,False
9,census_acs5_b25141_failures_2015_2025,9,4,False


## Observation grain

One county-year per ACS product in data tables; dictionary, metadata, coverage, and failure tables have their own supporting grains.

The checks below infer candidate keys from the raw columns. A repeated candidate key is
reported rather than silently removed because some provider tables legitimately contain
additional dimensions.

## Column types and meanings

In [3]:
schema_frames = []
for table in primary_tables:
    schema = con.execute(f"DESCRIBE raw.{qi(table)}").df()
    schema.insert(0, "table_name", table)
    schema["inferred_meaning"] = (
        schema["column_name"].str.replace("_", " ", regex=False)
        .str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)
        .str.strip()
    )
    schema_frames.append(schema)
schema_inventory = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
display(schema_inventory)

,table_name,column_name,column_type,null,key,default,extra,inferred_meaning
0,census_acs5_county_affordability_2012_2024,year,VARCHAR,YES,None,None,None,year
1,census_acs5_county_affordability_2012_2024,county_fips,VARCHAR,YES,None,None,None,county fips
2,census_acs5_county_affordability_2012_2024,state,VARCHAR,YES,None,None,None,state
3,census_acs5_county_affordability_2012_2024,county,VARCHAR,YES,None,None,None,county
4,census_acs5_county_affordability_2012_2024,NAME,VARCHAR,YES,None,None,None,NAME
...,...,...,...,...,...,...,...,...
4092,census_acs5_county_population_2015_2024,name,VARCHAR,YES,None,None,None,name
4093,census_acs5_county_population_2015_2024,total_population,VARCHAR,YES,None,None,None,total population
4094,census_acs5_county_population_2015_2024,total_population_moe,VARCHAR,YES,None,None,None,total population moe
4095,census_acs5_county_population_metadata,field,VARCHAR,YES,None,None,None,field


## Date and geographic coverage

In [4]:
coverage_rows = []
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"].tolist()
    row = {"table_name": table}
    for column in DATE_CANDIDATES:
        if column in columns:
            normalized_column = column.lower()
            if normalized_column == "year" or normalized_column.endswith("_year"):
                coverage_type = "INTEGER"
            elif normalized_column == "month" or normalized_column.endswith("_month"):
                coverage_type = "INTEGER"
            else:
                coverage_type = "TIMESTAMP"
            values = con.execute(
                f"SELECT min(try_cast({qi(column)} AS {coverage_type})), "
                f"max(try_cast({qi(column)} AS {coverage_type})) "
                f"FROM raw.{qi(table)}"
            ).fetchone()
            row[f"{column}_min"] = values[0]
            row[f"{column}_max"] = values[1]
    for column in GEO_CANDIDATES:
        if column in columns:
            row[f"{column}_distinct"] = con.execute(
                f"SELECT count(DISTINCT {qi(column)}) FROM raw.{qi(table)}"
            ).fetchone()[0]
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
display(coverage)

,table_name,year_min,year_max,state_distinct,county_distinct
0,census_acs5_county_affordability_2012_2024,2012.0,2024.0,52.0,332.0
1,census_acs5_county_dp02_2015_2015,2015.0,2015.0,52.0,325.0
2,census_acs5_county_dp02_2015_2016,2015.0,2016.0,52.0,325.0
3,census_acs5_county_dp02_2015_2018,2015.0,2018.0,52.0,325.0
4,census_acs5_county_dp02_2015_2024,2015.0,2024.0,52.0,330.0
5,census_acs5_county_dp02_metadata_2015_2015,NaN,NaN,NaN,NaN
6,census_acs5_county_dp02_metadata_2015_2016,NaN,NaN,NaN,NaN
7,census_acs5_county_dp02_metadata_2015_2018,NaN,NaN,NaN,NaN
8,census_acs5_county_dp02_metadata_2015_2024,NaN,NaN,NaN,NaN
9,census_acs5_county_dp03_2015_2024,2015.0,2024.0,52.0,330.0


## Missingness and suppression codes

In [5]:
missing_rows = []
suppression_rows = []
suppression_sql = ", ".join("?" for _ in SUPPRESSION_CODES)
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [table]
    ).df()["column_name"].tolist()
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    # Profile all columns for compact tables and the first 80 for unusually wide sources.
    for column in columns[:80]:
        null_count, blank_count = con.execute(
            f"SELECT count(*) FILTER (WHERE {qi(column)} IS NULL), "
            f"count(*) FILTER (WHERE trim(cast({qi(column)} AS VARCHAR))='') "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        missing_rows.append({
            "table_name": table, "column_name": column,
            "missing_count": null_count + blank_count,
            "missing_pct": (null_count + blank_count) / row_count * 100 if row_count else np.nan,
        })
        if SUPPRESSION_CODES:
            suppressed = con.execute(
                f"SELECT count(*) FROM raw.{qi(table)} "
                f"WHERE trim(cast({qi(column)} AS VARCHAR)) IN ({suppression_sql})",
                SUPPRESSION_CODES,
            ).fetchone()[0]
            if suppressed:
                suppression_rows.append({
                    "table_name": table, "column_name": column,
                    "suppression_or_sentinel_count": suppressed,
                })
missingness = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "table_name"], ascending=[False, True]
)
suppression = pd.DataFrame(suppression_rows)
display(missingness)
display(suppression if not suppression.empty else pd.DataFrame(
    {"result": ["No configured literal suppression codes were present in profiled columns; nulls remain material."]}
))

,table_name,column_name,missing_count,missing_pct
28,census_acs5_county_dp02_2015_2015,DP02_0001PE,3220,100.0
29,census_acs5_county_dp02_2015_2015,DP02_0001PM,3220,100.0
84,census_acs5_county_dp02_2015_2015,DP02_0015PE,3220,100.0
85,census_acs5_county_dp02_2015_2015,DP02_0015PM,3220,100.0
88,census_acs5_county_dp02_2015_2015,DP02_0016PE,3220,100.0
...,...,...,...,...
624,census_acs5_county_population_2015_2024,county,0,0.0
625,census_acs5_county_population_2015_2024,name,0,0.0
626,census_acs5_county_population_2015_2024,total_population,0,0.0
628,census_acs5_county_population_metadata,field,0,0.0


,result
0,No configured literal suppression codes were p...


## Duplicate or invalid keys

In [6]:
key_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    keys = next((candidate for candidate in KEY_CANDIDATES if set(candidate).issubset(columns)), [])
    if not keys:
        key_rows.append({"table_name": table, "candidate_key": None,
                         "duplicate_key_groups": np.nan, "invalid_key_rows": np.nan})
        continue
    key_expr = ", ".join(qi(column) for column in keys)
    invalid = " OR ".join(
        f"{qi(column)} IS NULL OR trim(cast({qi(column)} AS VARCHAR))=''" for column in keys
    )
    duplicate_groups = con.execute(
        f"SELECT count(*) FROM (SELECT {key_expr}, count(*) n "
        f"FROM raw.{qi(table)} GROUP BY {key_expr} HAVING count(*) > 1)"
    ).fetchone()[0]
    invalid_rows = con.execute(
        f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}"
    ).fetchone()[0]
    key_rows.append({"table_name": table, "candidate_key": " + ".join(keys),
                     "duplicate_key_groups": duplicate_groups,
                     "invalid_key_rows": invalid_rows})
key_quality = pd.DataFrame(key_rows)
display(key_quality)

,table_name,candidate_key,duplicate_key_groups,invalid_key_rows
0,census_acs5_county_affordability_2012_2024,year + state + county,0.0,0.0
1,census_acs5_county_dp02_2015_2015,year + state + county,0.0,0.0
2,census_acs5_county_dp02_2015_2016,year + state + county,0.0,0.0
3,census_acs5_county_dp02_2015_2018,year + state + county,0.0,0.0
4,census_acs5_county_dp02_2015_2024,year + state + county,0.0,0.0
5,census_acs5_county_dp02_metadata_2015_2015,None,NaN,NaN
6,census_acs5_county_dp02_metadata_2015_2016,None,NaN,NaN
7,census_acs5_county_dp02_metadata_2015_2018,None,NaN,NaN
8,census_acs5_county_dp02_metadata_2015_2024,None,NaN,NaN
9,census_acs5_county_dp03_2015_2024,year + state + county,0.0,0.0


## Numeric ranges and suspicious values

In [7]:
numeric_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    for column in [name for name in NUMERIC_HINTS if name in columns]:
        numeric = (
            f"try_cast(replace(trim(cast({qi(column)} AS VARCHAR)), ',', '') AS DOUBLE)"
        )
        result = con.execute(
            f"SELECT count(*) FILTER (WHERE {numeric} IS NOT NULL), "
            f"min({numeric}), max({numeric}), "
            f"count(*) FILTER (WHERE {numeric} < 0) "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        numeric_rows.append({
            "table_name": table, "column_name": column,
            "numeric_count": result[0], "minimum": result[1],
            "maximum": result[2], "negative_count": result[3],
            "review_flag": (
                "review negative values/sentinels" if result[3] else
                "review extreme min/max against provider definition"
            ),
        })
numeric_ranges = pd.DataFrame(numeric_rows)
display(numeric_ranges)

,table_name,column_name,numeric_count,minimum,maximum,negative_count,review_flag
0,census_acs5_county_affordability_2012_2024,year,41870,2012.0,2024.0,0,review extreme min/max against provider defini...
1,census_acs5_county_affordability_2012_2024,median_household_income,41862,10499.0,181765.0,0,review extreme min/max against provider defini...
2,census_acs5_county_affordability_2012_2024,median_home_value,41834,18700.0,1633900.0,0,review extreme min/max against provider defini...
3,census_acs5_county_dp02_2015_2015,year,3220,2015.0,2015.0,0,review extreme min/max against provider defini...
4,census_acs5_county_dp02_2015_2016,year,6440,2015.0,2016.0,0,review extreme min/max against provider defini...
5,census_acs5_county_dp02_2015_2018,year,12880,2015.0,2018.0,0,review extreme min/max against provider defini...
6,census_acs5_county_dp02_2015_2024,year,32208,2015.0,2024.0,0,review extreme min/max against provider defini...
7,census_acs5_county_dp03_2015_2024,year,32208,2015.0,2024.0,0,review extreme min/max against provider defini...
8,census_acs5_county_dp04_2015_2024,year,32208,2015.0,2024.0,0,review extreme min/max against provider defini...
9,census_acs5_county_dp05_2015_2024,year,32208,2015.0,2024.0,0,review extreme min/max against provider defini...


## Source-specific limitations

ACS five-year estimates represent rolling periods, margins of error vary by county and measure, variable labels can change across releases, and suppressed or unavailable values must not be treated as zero.

## Downstream readiness

**Assessment: PASS WITH LIMITATIONS when county-year keys are valid and suppression values are interpreted as missing. Downstream marts must preserve margins of error and release-year provenance.**

This assessment is conditional on the displayed inventories and checks. The normalized
`mart.*` builders—not this notebook—own parsing, suppression handling, geographic
resolution, deduplication, and downstream transformations.

In [8]:
con.close()